# Lab: Building a 2-Layer Neural Network from Scratch

*A graded-assignment-style lab inspired by the Week 3 programming assignment in DeepLearning.AI's "Calculus for Machine Learning and Data Science" course.*

This is **not** a copy of the course assignment -- it's an original lab that puts together every piece of calculus you have studied in this course:

| Concept | Where you learned it |
|---|---|
| Sigmoid derivative $\sigma'(z) = \hat{y}(1-\hat{y})$ | Week 1 Lesson 1 |
| $\tanh$ derivative $1 - \tanh^2(z)$ | Week 3 Part 1 Quiz |
| Log-loss and $\partial L/\partial z = \hat{y} - y$ | Week 3 Part 1 Quiz |
| Chain rule through multiple layers | Week 3 Part 1 Quiz Part B |
| Gradient descent weight update | Week 2 & Week 3 |

You will build everything yourself from NumPy operations -- no ML libraries for the neural network logic. At the end you will achieve **>95% accuracy** on a non-linearly separable dataset that a single neuron cannot solve at all.

**How to use this notebook:**
1. Fill in every `### START CODE HERE ###` block.
2. Run the test cell immediately after -- it will tell you if your implementation is correct.
3. Do **not** peek at the Solutions section until you have tried everything.

---

**A new concept you will encounter here (not in the course lectures):**
At the end of the lab there is a bonus section on the **symmetry breaking problem**: why initialising all weights to zero causes the network to fail completely regardless of how long you train it, and why this is a direct consequence of the chain rule you derived.


## Table of Contents
1. Imports and Dataset
2. Network Architecture
3. Exercise 1 -- Activation Functions
4. Exercise 2 -- Initialise Parameters
5. Exercise 3 -- Forward Propagation
6. Exercise 4 -- Compute the Cost
7. Exercise 5 -- Backward Propagation
8. Exercise 6 -- Update Parameters
9. Exercise 7 -- The Training Loop
10. Exercise 8 -- Predict and Evaluate
11. Bonus -- The Symmetry Breaking Problem
12. Wrap-up
13. Solutions


## 1. Imports and Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (7, 5)


In [ ]:
# Generate dataset
X_raw, Y_raw = make_moons(n_samples=400, noise=0.2, random_state=42)
X_tr, X_te, Y_tr, Y_te = train_test_split(X_raw, Y_raw, test_size=0.2, random_state=42)

# Reshape to (n_features, m_examples) -- the column-per-example convention used throughout
X_train = X_tr.T          # shape (2, 320)
X_test  = X_te.T          # shape (2, 80)
Y_train = Y_tr.reshape(1, -1)   # shape (1, 320)
Y_test  = Y_te.reshape(1, -1)   # shape (1, 80)

print("X_train:", X_train.shape, "  Y_train:", Y_train.shape)
print("X_test: ", X_test.shape,  "  Y_test: ", Y_test.shape)
print("Class balance (train): {:.0f}% class 0 / {:.0f}% class 1".format(
    100*np.mean(Y_train==0), 100*np.mean(Y_train==1)))


In [ ]:
# Visualise the dataset
plt.scatter(X_raw[:, 0], X_raw[:, 1], c=Y_raw, cmap='RdBu', edgecolors='k', s=30)
plt.title("Dataset: two interleaved half-moons")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$")
plt.colorbar(label="class")
plt.show()
print("This dataset is NOT linearly separable -- you need a hidden layer to classify it correctly.")


## 2. Network Architecture

You will build a **2-layer neural network** (one hidden layer + one output layer):

```
Input (2)  -->  Hidden layer (8 neurons, tanh)  -->  Output (1 neuron, sigmoid)  -->  loss
```

Concretely, for $m$ training examples stacked as columns in $X$:

**Forward pass:**
$$Z^{[1]} = W^{[1]}X + b^{[1]}, \quad A^{[1]} = \tanh(Z^{[1]})$$
$$Z^{[2]} = W^{[2]}A^{[1]} + b^{[2]}, \quad A^{[2]} = \sigma(Z^{[2]})$$

**Cost (binary cross-entropy over $m$ examples):**
$$\mathcal{L} = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log a^{[2](i)} + (1-y^{(i)})\log(1-a^{[2](i)})\right]$$

**Backward pass (using the chain rule results from your quizzes):**
$$dZ^{[2]} = A^{[2]} - Y \qquad\leftarrow\;\partial\mathcal{L}/\partial z = \hat{y}-y\text{ from Quiz W3P1}$$
$$dW^{[2]} = \frac{1}{m}dZ^{[2]}(A^{[1]})^T, \quad db^{[2]} = \frac{1}{m}\sum dZ^{[2]}$$
$$dA^{[1]} = (W^{[2]})^T dZ^{[2]}$$
$$dZ^{[1]} = dA^{[1]} \odot (1-(A^{[1]})^2) \qquad\leftarrow\;\tanh'(z)=1-\tanh^2(z)\text{ from Quiz W3P1}$$
$$dW^{[1]} = \frac{1}{m}dZ^{[1]}X^T, \quad db^{[1]} = \frac{1}{m}\sum dZ^{[1]}$$


In [ ]:
# Network dimensions -- fixed for this lab
N_X = 2    # input features
N_H = 8    # hidden neurons
N_Y = 1    # output neurons


## 3. Exercise 1 -- Activation Functions

Implement the three functions needed for forward and backward propagation.
All three should work element-wise on NumPy arrays.


In [ ]:
# GRADED FUNCTIONS: sigmoid, tanh_activation, tanh_prime

def sigmoid(z):
    # Returns the sigmoid of z: 1 / (1 + e^{-z})
    ### START CODE HERE ### (~ 1 line)
    result = None
    ### END CODE HERE ###
    return result

def tanh_activation(z):
    # Returns tanh(z)  [use np.tanh]
    ### START CODE HERE ### (~ 1 line)
    result = None
    ### END CODE HERE ###
    return result

def tanh_prime(z):
    # Returns the derivative of tanh at z: 1 - tanh^2(z)
    # (the result you proved in the Week 3 Part 1 Quiz, Q7)
    ### START CODE HERE ### (~ 1 line)
    result = None
    ### END CODE HERE ###
    return result


In [ ]:
# Test cell -- Exercise 1
z_test = np.array([-2., -1., 0., 1., 2.])

assert np.allclose(sigmoid(z_test),
    [0.1192, 0.2689, 0.5, 0.7311, 0.8808], atol=1e-3), "sigmoid failed"

assert np.allclose(tanh_activation(z_test),
    [-0.9640, -0.7616, 0., 0.7616, 0.9640], atol=1e-3), "tanh failed"

assert np.allclose(tanh_prime(z_test),
    [0.0707, 0.4200, 1.0, 0.4200, 0.0707], atol=1e-3), "tanh_prime failed"

# Verify tanh_prime = 1 - tanh^2  (the identity you proved)
assert np.allclose(tanh_prime(z_test),
    1 - tanh_activation(z_test)**2, atol=1e-9), "tanh_prime identity failed"

print("Exercise 1 looks correct!")


## 4. Exercise 2 -- Initialise Parameters

Initialise the weight matrices and bias vectors for both layers.

**Shapes:**
- $W^{[1]}$: $(n_h,\, n_x)$, initialised to small random values (multiply by `0.01`)
- $b^{[1]}$: $(n_h,\, 1)$, initialised to zeros
- $W^{[2]}$: $(n_y,\, n_h)$, initialised to small random values (multiply by `0.01`)
- $b^{[2]}$: $(n_y,\, 1)$, initialised to zeros

Use `np.random.randn(...)` for the random weights. The `0.01` scaling keeps the initial $z$ values small so activations start in the sensitive (non-saturated) region of tanh and sigmoid.


In [ ]:
# GRADED FUNCTION: init_params

def init_params(n_x, n_h, n_y, seed=42):
    np.random.seed(seed)
    ### START CODE HERE ###
    W1 = None   # shape (n_h, n_x)
    b1 = None   # shape (n_h, 1)
    W2 = None   # shape (n_y, n_h)
    b2 = None   # shape (n_y, 1)
    ### END CODE HERE ###
    return W1, b1, W2, b2


In [ ]:
# Test cell -- Exercise 2
W1, b1, W2, b2 = init_params(N_X, N_H, N_Y)

assert W1.shape == (N_H, N_X), f"W1 shape should be ({N_H},{N_X}), got {W1.shape}"
assert b1.shape == (N_H, 1),   f"b1 shape should be ({N_H},1), got {b1.shape}"
assert W2.shape == (N_Y, N_H), f"W2 shape should be ({N_Y},{N_H}), got {W2.shape}"
assert b2.shape == (N_Y, 1),   f"b2 shape should be ({N_Y},1), got {b2.shape}"
assert np.all(b1 == 0) and np.all(b2 == 0), "biases should be zero"
assert np.max(np.abs(W1)) < 0.1 and np.max(np.abs(W2)) < 0.1, "weights should be small (scaled by 0.01)"

print("W1:", W1.shape, "  b1:", b1.shape)
print("W2:", W2.shape, "  b2:", b2.shape)
print("Exercise 2 looks correct!")


## 5. Exercise 3 -- Forward Propagation

Implement the forward pass through both layers. Return both $A^{[2]}$ (the predictions) and a `cache` tuple containing $(Z^{[1]}, A^{[1]}, Z^{[2]}, A^{[2]})$ -- you will need these cached values during backward propagation.


In [ ]:
# GRADED FUNCTION: forward_propagation

def forward_propagation(X, W1, b1, W2, b2):
    ### START CODE HERE ###
    Z1 = None           # (N_H, m)
    A1 = None           # (N_H, m)  -- tanh activation
    Z2 = None           # (N_Y, m)
    A2 = None           # (N_Y, m)  -- sigmoid activation (predictions)
    ### END CODE HERE ###
    cache = (Z1, A1, Z2, A2)
    return A2, cache


In [ ]:
# Test cell -- Exercise 3
np.random.seed(1)
X_tmp = np.random.randn(2, 5)
W1_tmp, b1_tmp, W2_tmp, b2_tmp = init_params(2, 4, 1, seed=1)

A2_tmp, cache_tmp = forward_propagation(X_tmp, W1_tmp, b1_tmp, W2_tmp, b2_tmp)
Z1_tmp, A1_tmp, Z2_tmp, A2_tmp2 = cache_tmp

assert A2_tmp.shape == (1, 5),  f"A2 shape should be (1,5), got {A2_tmp.shape}"
assert A1_tmp.shape == (4, 5),  f"A1 shape should be (4,5), got {A1_tmp.shape}"
assert np.all(A2_tmp >= 0) and np.all(A2_tmp <= 1), "A2 (sigmoid output) must be in [0,1]"
assert np.all(A1_tmp >= -1) and np.all(A1_tmp <= 1), "A1 (tanh output) must be in [-1,1]"
assert np.allclose(A2_tmp, A2_tmp2), "A2 in return value and cache should match"

print("A2 (predictions):", np.round(A2_tmp, 4))
print("Exercise 3 looks correct!")


## 6. Exercise 4 -- Compute the Cost

Implement the binary cross-entropy cost averaged over $m$ examples:

$$\mathcal{L} = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log a^{[2](i)} + (1-y^{(i)})\log(1-a^{[2](i)})\right]$$

*Tip: add a tiny constant `1e-8` inside each `np.log(...)` to avoid $\log(0)$ numerical issues.*


In [ ]:
# GRADED FUNCTION: compute_cost

def compute_cost(A2, Y):
    m = Y.shape[1]
    ### START CODE HERE ### (~ 2 lines)
    cost = None
    ### END CODE HERE ###
    return float(cost)


In [ ]:
# Test cell -- Exercise 4
# When predictions are 50/50 (A2=0.5 everywhere), cost should be ln(2) ~ 0.6931
A2_half = np.full((1, 10), 0.5)
Y_half  = np.array([[1,0,1,0,1,0,1,0,1,0]])
cost_half = compute_cost(A2_half, Y_half)
assert np.isclose(cost_half, np.log(2), atol=1e-3), f"Expected ~0.6931, got {cost_half:.4f}"

# When predictions are perfect (up to numerical precision), cost should be near 0
A2_perfect = np.array([[0.9999, 0.0001, 0.9999]])
Y_perfect  = np.array([[1, 0, 1]])
cost_perfect = compute_cost(A2_perfect, Y_perfect)
assert cost_perfect < 0.01, f"Cost with near-perfect predictions should be near 0, got {cost_perfect:.4f}"

print(f"Cost at 50/50 predictions: {cost_half:.4f}  (expected ~0.6931)")
print(f"Cost at near-perfect predictions: {cost_perfect:.6f}  (expected ~0)")
print("Exercise 4 looks correct!")


## 7. Exercise 5 -- Backward Propagation

This is the heart of the lab. Implement the full backward pass using the chain rule.

The formulas below follow directly from the derivations in your quizzes:

| Quantity | Formula | Quiz connection |
|---|---|---|
| $dZ^{[2]}$ | $A^{[2]} - Y$ | $\partial\mathcal{L}/\partial z = \hat{y}-y$ (W3P1 Q3) |
| $dW^{[2]}$ | $\frac{1}{m}\,dZ^{[2]}(A^{[1]})^T$ | chain rule $\partial L/\partial w = (\hat{y}-y)\cdot x$ (W3P1 Q4) |
| $db^{[2]}$ | $\frac{1}{m}\sum dZ^{[2]}$ (keepdims) | same |
| $dA^{[1]}$ | $(W^{[2]})^T\,dZ^{[2]}$ | multi-layer chain rule (W3P1 Q9) |
| $dZ^{[1]}$ | $dA^{[1]} \odot (1-(A^{[1]})^2)$ | $\tanh'(z)=1-\tanh^2(z)$ (W3P1 Q7) |
| $dW^{[1]}$ | $\frac{1}{m}\,dZ^{[1]}X^T$ | |
| $db^{[1]}$ | $\frac{1}{m}\sum dZ^{[1]}$ (keepdims) | |

Note: $dA^{[1]} \odot (1-(A^{[1]})^2)$ uses element-wise multiplication (`*` in NumPy) -- this is the chain rule linking $dA^{[1]}$ back through the tanh activation to $dZ^{[1]}$.


In [ ]:
# GRADED FUNCTION: backward_propagation

def backward_propagation(X, Y, W2, cache):
    m = Y.shape[1]
    Z1, A1, Z2, A2 = cache

    ### START CODE HERE ###
    dZ2 = None          # (N_Y, m)
    dW2 = None          # (N_Y, N_H)
    db2 = None          # (N_Y, 1)
    dA1 = None          # (N_H, m)
    dZ1 = None          # (N_H, m)  -- use tanh_prime or equivalently (1 - A1**2)
    dW1 = None          # (N_H, N_X)
    db1 = None          # (N_H, 1)
    ### END CODE HERE ###

    grads = {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
    return grads


In [ ]:
# Test cell -- Exercise 5
np.random.seed(2)
X_tmp2 = np.random.randn(2, 6)
Y_tmp2 = np.array([[1,0,1,0,1,0]])
W1_t, b1_t, W2_t, b2_t = init_params(2, 4, 1, seed=2)
A2_t, cache_t = forward_propagation(X_tmp2, W1_t, b1_t, W2_t, b2_t)
grads_t = backward_propagation(X_tmp2, Y_tmp2, W2_t, cache_t)

assert grads_t['dW1'].shape == (4, 2), f"dW1 shape wrong: {grads_t['dW1'].shape}"
assert grads_t['db1'].shape == (4, 1), f"db1 shape wrong: {grads_t['db1'].shape}"
assert grads_t['dW2'].shape == (1, 4), f"dW2 shape wrong: {grads_t['dW2'].shape}"
assert grads_t['db2'].shape == (1, 1), f"db2 shape wrong: {grads_t['db2'].shape}"

# Numerical gradient check on dW2
eps = 1e-5
W2_plus  = W2_t.copy(); W2_plus[0,0] += eps
W2_minus = W2_t.copy(); W2_minus[0,0] -= eps
A2p, _ = forward_propagation(X_tmp2, W1_t, b1_t, W2_plus,  b2_t)
A2m, _ = forward_propagation(X_tmp2, W1_t, b1_t, W2_minus, b2_t)
dW2_numerical = (compute_cost(A2p, Y_tmp2) - compute_cost(A2m, Y_tmp2)) / (2*eps)
assert np.isclose(grads_t['dW2'][0,0], dW2_numerical, atol=1e-5),     f"dW2[0,0] mismatch: analytic={grads_t['dW2'][0,0]:.6f}, numerical={dW2_numerical:.6f}"

print("Gradient shapes:")
for k, v in grads_t.items():
    print(f"  {k}: {v.shape}")
print("Numerical gradient check passed!")
print("Exercise 5 looks correct!")


## 8. Exercise 6 -- Update Parameters

Apply one step of gradient descent to all four parameter arrays:

$$W^{[l]} \leftarrow W^{[l]} - \alpha\,dW^{[l]}, \qquad b^{[l]} \leftarrow b^{[l]} - \alpha\,db^{[l]}$$


In [ ]:
# GRADED FUNCTION: update_params

def update_params(W1, b1, W2, b2, grads, learning_rate):
    ### START CODE HERE ###
    W1 = None
    b1 = None
    W2 = None
    b2 = None
    ### END CODE HERE ###
    return W1, b1, W2, b2


In [ ]:
# Test cell -- Exercise 6
W1_u, b1_u, W2_u, b2_u = init_params(2, 3, 1, seed=3)
grads_u = {'dW1': np.ones((3,2)), 'db1': np.ones((3,1)),
           'dW2': np.ones((1,3)), 'db2': np.ones((1,1))}

W1_new, b1_new, W2_new, b2_new = update_params(W1_u, b1_u, W2_u, b2_u, grads_u, learning_rate=0.1)

assert np.allclose(W1_new, W1_u - 0.1), "W1 update incorrect"
assert np.allclose(b1_new, b1_u - 0.1), "b1 update incorrect"
assert np.allclose(W2_new, W2_u - 0.1), "W2 update incorrect"
assert np.allclose(b2_new, b2_u - 0.1), "b2 update incorrect"
print("Exercise 6 looks correct!")


## 9. Exercise 7 -- The Training Loop

Assemble all previous functions into the full training loop:

```
for each iteration:
    1. forward propagation  -> A2, cache
    2. compute cost         -> L
    3. backward propagation -> grads
    4. update parameters    -> W1, b1, W2, b2
```

Return the trained parameters and the list of costs (recorded every 1000 iterations for plotting).


In [ ]:
# GRADED FUNCTION: nn_model

def nn_model(X, Y, n_h, learning_rate=0.5, num_iterations=10000, print_cost=True):
    np.random.seed(42)
    n_x = X.shape[0]
    n_y = Y.shape[0]

    W1, b1, W2, b2 = init_params(n_x, n_h, n_y)
    costs = []

    for i in range(num_iterations):
        ### START CODE HERE ###
        # 1. Forward propagation
        A2, cache = None, None
        # 2. Compute cost
        cost = None
        # 3. Backward propagation
        grads = None
        # 4. Update parameters
        W1, b1, W2, b2 = None, None, None, None
        ### END CODE HERE ###

        if i % 1000 == 0:
            costs.append(cost)
            if print_cost:
                print(f"Iteration {i:5d}  |  cost = {cost:.4f}")

    params = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}
    return params, costs


In [ ]:
# Train the network
print("Training 2-layer neural network on the moons dataset...")
print("-" * 50)
params, costs = nn_model(X_train, Y_train, n_h=N_H, learning_rate=0.5, num_iterations=10000)
print("-" * 50)
print("Training complete!")


In [ ]:
# Plot the learning curve
plt.plot(np.arange(len(costs)) * 1000, costs, 'b.-')
plt.xlabel("Iteration"); plt.ylabel("Cost")
plt.title("Learning curve -- cost should decrease steadily")
plt.grid(True)
plt.show()

assert costs[0] > costs[-1], "Cost should decrease during training!"
assert costs[-1] < 0.15, f"Final cost {costs[-1]:.4f} seems too high -- check your implementation"
print(f"Initial cost: {costs[0]:.4f}")
print(f"Final cost:   {costs[-1]:.4f}")
print("Exercise 7 looks correct!")


## 10. Exercise 8 -- Predict and Evaluate

Implement `predict`, which runs a forward pass and thresholds $A^{[2]}$ at 0.5 to produce binary class labels (0 or 1).


In [ ]:
# GRADED FUNCTION: predict

def predict(X, params):
    W1, b1, W2, b2 = params['W1'], params['b1'], params['W2'], params['b2']
    ### START CODE HERE ###
    A2, _ = None, None      # forward pass
    predictions = None       # 1 where A2 > 0.5, else 0
    ### END CODE HERE ###
    return predictions


In [ ]:
# Test cell + accuracy
train_preds = predict(X_train, params)
test_preds  = predict(X_test,  params)

train_acc = float(np.mean(train_preds == Y_train))
test_acc  = float(np.mean(test_preds  == Y_test))

assert train_preds.shape == Y_train.shape, "prediction shape should match Y_train"
assert set(np.unique(train_preds)).issubset({0,1}), "predictions must be 0 or 1"
assert train_acc > 0.90, f"Train accuracy {train_acc:.2%} too low -- check your model"
assert test_acc  > 0.90, f"Test accuracy {test_acc:.2%} too low -- possible overfitting"

print(f"Train accuracy: {train_acc:.2%}")
print(f"Test  accuracy: {test_acc:.2%}")
print("Exercise 8 looks correct!")


In [ ]:
# Visualise the decision boundary
def plot_decision_boundary(X, Y, params, title):
    x_min, x_max = X[0].min() - 0.5, X[0].max() + 0.5
    y_min, y_max = X[1].min() - 0.5, X[1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()].T
    Z = predict(grid, params).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    plt.scatter(X[0], X[1], c=Y[0], cmap='RdBu', edgecolors='k', s=25)
    plt.title(title)
    plt.xlabel("$x_1$"); plt.ylabel("$x_2$")

plot_decision_boundary(X_test, Y_test, params,
    f"Decision boundary (test accuracy = {test_acc:.1%})")
plt.show()


## 11. Bonus -- The Symmetry Breaking Problem

> **The new concept for this lab:** what goes wrong if you initialise all weights to zero?

In Exercise 2 you initialised the weights with small *random* values. Let's see what happens when we initialise everything to **zero** instead.


In [ ]:
# Train with zero initialisation
def init_params_zeros(n_x, n_h, n_y):
    # All weights and biases set to zero
    W1 = np.zeros((n_h, n_x))
    b1 = np.zeros((n_h, 1))
    W2 = np.zeros((n_y, n_h))
    b2 = np.zeros((n_y, 1))
    return W1, b1, W2, b2

def nn_model_zeros(X, Y, n_h, learning_rate=0.5, num_iterations=10000):
    n_x, n_y = X.shape[0], Y.shape[0]
    W1, b1, W2, b2 = init_params_zeros(n_x, n_h, n_y)
    costs = []
    for i in range(num_iterations):
        A2, cache = forward_propagation(X, W1, b1, W2, b2)
        cost = compute_cost(A2, Y)
        grads = backward_propagation(X, Y, W2, cache)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, grads, learning_rate)
        if i % 1000 == 0:
            costs.append(cost)
    params_z = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}
    return params_z, costs, W1

print("Training with zero initialisation...")
params_z, costs_z, W1_final_z = nn_model_zeros(X_train, Y_train, n_h=N_H)
zero_train_acc = float(np.mean(predict(X_train, params_z) == Y_train))
print(f"Zero-init train accuracy: {zero_train_acc:.2%}  (no better than random guessing!)")


In [ ]:
# Show that ALL hidden neurons learned exactly the same thing
print("Are all rows of W1 identical after zero-init training?")
all_identical = all(np.allclose(W1_final_z[0], W1_final_z[i]) for i in range(N_H))
print(f"  Yes: {all_identical}")
print()
print("W1[0] (first hidden neuron weights):", np.round(W1_final_z[0], 6))
print("W1[1] (second hidden neuron weights):", np.round(W1_final_z[1], 6))


In [ ]:
# Side-by-side: random init vs zero init
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plt.sca(axes[0])
plot_decision_boundary(X_test, Y_test, params,
    f"Random init  |  test acc = {test_acc:.1%}")

plt.sca(axes[1])
plot_decision_boundary(X_test, Y_test, params_z,
    f"Zero init  |  test acc = {zero_train_acc:.1%}")

plt.tight_layout()
plt.show()


### Why does zero initialisation fail?

The answer is a direct consequence of the backpropagation you derived.

When all weights are zero, every hidden neuron receives the same input to its activation:
$$Z^{[1]} = W^{[1]}X + b^{[1]} = 0 \quad \Rightarrow \quad A^{[1]} = \tanh(0) = 0 \text{ for all neurons}$$

Now look at the gradient update for $W^{[1]}$:
$$dZ^{[1]} = dA^{[1]} \odot \underbrace{(1 - (A^{[1]})^2)}_{= 1 \text{ since } A^{[1]}=0} = dA^{[1]}$$
$$dA^{[1]} = (W^{[2]})^T dZ^{[2]} = 0 \cdot dZ^{[2]} = 0$$

So $dW^{[1]} = 0$ -- **all weights remain zero forever**. The network never learns.

And even if $W^{[2]}$ starts non-zero: every row of $W^{[1]}$ receives an identical gradient at every step (since all neurons see the same activations), so they remain permanently identical. Eight neurons with zero weights behave **exactly like one neuron with zero weights** -- no matter how long you train.

This is why random initialisation is essential: it **breaks the symmetry**, giving each neuron a different starting point so that backpropagation can specialise them to detect different features.


## 12. Wrap-up

If all tests passed, you have built a fully functional 2-layer neural network **from scratch using only NumPy**. Here is what every piece you implemented corresponds to in the calculus you studied:

| Implementation | Calculus behind it |
|---|---|
| `sigmoid`, `tanh_activation`, `tanh_prime` | Derivatives of activation functions (W1L1, W3P1 Q6-Q7) |
| `forward_propagation` | Composition of functions; the forward pass |
| `compute_cost` | Log-loss / binary cross-entropy (W3P1 Q2) |
| `backward_propagation` -- $dZ^{[2]}=A^{[2]}-Y$ | $\partial L/\partial z = \hat{y}-y$ (W3P1 Q3) |
| `backward_propagation` -- $dZ^{[1]}=dA^{[1]}\odot(1-(A^{[1]})^2)$ | Multi-layer chain rule (W3P1 Q9-Q11) |
| `update_params` | Gradient descent weight update (W2, W3) |
| Zero-init failure | Consequence of chain rule: zero activations -> zero gradients |


## 13. Solutions

*Try everything on your own first!*


In [ ]:
# ----- Solution: Exercise 1 -----
def sigmoid(z):
    result = 1 / (1 + np.exp(-z))
    return result

def tanh_activation(z):
    result = np.tanh(z)
    return result

def tanh_prime(z):
    result = 1 - np.tanh(z)**2
    return result


In [ ]:
# ----- Solution: Exercise 2 -----
def init_params(n_x, n_h, n_y, seed=42):
    np.random.seed(seed)
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    return W1, b1, W2, b2


In [ ]:
# ----- Solution: Exercise 3 -----
def forward_propagation(X, W1, b1, W2, b2):
    Z1 = W1 @ X + b1
    A1 = tanh_activation(Z1)
    Z2 = W2 @ A1 + b2
    A2 = sigmoid(Z2)
    cache = (Z1, A1, Z2, A2)
    return A2, cache


In [ ]:
# ----- Solution: Exercise 4 -----
def compute_cost(A2, Y):
    m = Y.shape[1]
    cost = -1/m * np.sum(Y * np.log(A2 + 1e-8) + (1 - Y) * np.log(1 - A2 + 1e-8))
    return float(cost)


In [ ]:
# ----- Solution: Exercise 5 -----
def backward_propagation(X, Y, W2, cache):
    m = Y.shape[1]
    Z1, A1, Z2, A2 = cache
    dZ2 = A2 - Y
    dW2 = 1/m * dZ2 @ A1.T
    db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)
    dA1 = W2.T @ dZ2
    dZ1 = dA1 * (1 - A1**2)      # tanh'(z) = 1 - tanh^2(z) = 1 - A1^2
    dW1 = 1/m * dZ1 @ X.T
    db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)
    grads = {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
    return grads


In [ ]:
# ----- Solution: Exercise 6 -----
def update_params(W1, b1, W2, b2, grads, learning_rate):
    W1 = W1 - learning_rate * grads['dW1']
    b1 = b1 - learning_rate * grads['db1']
    W2 = W2 - learning_rate * grads['dW2']
    b2 = b2 - learning_rate * grads['db2']
    return W1, b1, W2, b2


In [ ]:
# ----- Solution: Exercise 7 -----
def nn_model(X, Y, n_h, learning_rate=0.5, num_iterations=10000, print_cost=True):
    np.random.seed(42)
    n_x = X.shape[0]
    n_y = Y.shape[0]
    W1, b1, W2, b2 = init_params(n_x, n_h, n_y)
    costs = []
    for i in range(num_iterations):
        A2, cache = forward_propagation(X, W1, b1, W2, b2)
        cost = compute_cost(A2, Y)
        grads = backward_propagation(X, Y, W2, cache)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, grads, learning_rate)
        if i % 1000 == 0:
            costs.append(cost)
            if print_cost:
                print(f"Iteration {i:5d}  |  cost = {cost:.4f}")
    params = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}
    return params, costs


In [ ]:
# ----- Solution: Exercise 8 -----
def predict(X, params):
    W1, b1, W2, b2 = params['W1'], params['b1'], params['W2'], params['b2']
    A2, _ = forward_propagation(X, W1, b1, W2, b2)
    predictions = (A2 > 0.5).astype(int)
    return predictions
